# CENG468 Veri Madenciliği · Hafta 9
## İçerik tabanlı öneri: "bu filmi beğendin, şuna benziyor"
Öğr. Gör. Dr. Duygu TOPALOĞLU · 2026-2027 Güz

---
### Bu derste nasıl çalışacağız?
İşaretler önceki defterlerle aynı:

| İşaret | Ne yapıyorsun? |
|---|---|
| 🔮 **Tahmin et** | Çalıştırmadan önce çıktıyı tahmin et ve yaz. |
| 🔧 **Değiştir** | Çalışan kodu küçük bir farkla uyarla. |
| ✍️ **Kendin yaz** | Boş hücreye kodu kendin yaz. |
| 🔓 **Çözüm** | Önce dene, sonra aç. |

**Bugünün sorusu:** Sepet analizi "bunu alan şunu da aldı" dedi; ürünlerin **ne olduğuna** hiç bakmadı. Peki ürünlerin kendisine bakarak öneri yapılabilir mi? Toy Story'yi beğenen birine, Toy Story'ye **benzeyen** filmleri önerelim. Benzerliği filmlerin **türlerinden** ölçeceğiz.

**Veri:** MovieLens ml-latest-small (`Veri/movielens/BENIOKU.md`). Bu hafta ve sonrasında sınıf içi verimiz bu.

---
## Bölüm 0: Veriyi tanı (5 dk)
`movies.csv` üç sütun: film numarası, ad, türler. Türler tek hücrede `|` ile ayrılmış.

In [ ]:
import os
import numpy as np
import pandas as pd

VERI = os.path.join("Veri", "movielens", "ml-latest-small")
if not os.path.exists(os.path.join(VERI, "movies.csv")):           # zip bir kat fazla açıldıysa
    VERI = os.path.join(VERI, "ml-latest-small")

filmler = pd.read_csv(os.path.join(VERI, "movies.csv"))
puanlar = pd.read_csv(os.path.join(VERI, "ratings.csv"))
print(filmler.shape, puanlar.shape)

# Veri kirliliği 1: birkaç film adı iki kez geçiyor (farklı movieId, aynı ad).
# Adı anahtar olarak kullanacağımız için tekrarları atıyoruz.
tekrar = filmler["title"].duplicated(keep=False)
print("Tekrar eden adlar:", filmler.loc[tekrar, "title"].unique())
filmler = filmler.drop_duplicates("title").reset_index(drop=True)
filmler.head()

**🔮 Tahmin et:** `filmler["genres"]` sütununda kaç **farklı tür** var? (Toy Story'nin satırında beş tür yazıyor; ama farklı tür sayısı kaç?) Nasıl sayarsın?

*Tahminim:* ...

In [ ]:
turler = filmler["genres"].str.get_dummies("|")   # her tür bir sütun, 1 = film bu türde
print(turler.shape)
turler.columns.tolist()

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Kod ne yapıyor?</b>

`str.get_dummies("|")`: hücredeki metni `|` işaretinden bölüp her parça için bir sütun açar; film o türdeyse 1, değilse 0. Sonuç 9.737 film × 20 sütun (ham dosyada 9.742 film; 5 tekrarlı ad atıldı). Sütunlardan biri `(no genres listed)`: türü yazılmamış filmler. Bu da bir veri kirliliği; içerik tabanlı yöntem, içerik yoksa çalışmaz (sepet analizinde böyle bir sorun yoktu).

Bu tabloya **öğe profili** (item profile) diyoruz: her film, 20 sayıdan oluşan bir **vektör**.

</div>

</details>

---
## Bölüm 1: Benzerlik elle (15 dk)
Bütün tabloya geçmeden beş filmle çalışalım. Sunumdaki örnek:

In [ ]:
secilen = [1, 2, 6, 10, 7]                   # Toy Story, Jumanji, Heat, GoldenEye, Sabrina
mini = turler.loc[filmler["movieId"].isin(secilen)]
mini.index = filmler.loc[mini.index, "title"].str.replace(r" \(\d{4}\)", "", regex=True)
mini = mini.loc[:, mini.sum() > 0]             # bu beş filmde hiç geçmeyen türleri gizle
mini

**🔮 Tahmin et:** Toy Story ile Jumanji'nin kaç **ortak** türü var? Toy Story ile GoldenEye'ın? Tablodan say.

*Tahminim:* ...

İki vektörün benzerliğini **kosinüs benzerliği** (cosine similarity) ile ölçeceğiz:

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|} = \frac{\sum_i a_i b_i}{\sqrt{\sum_i a_i^2}\,\sqrt{\sum_i b_i^2}}$$

0/1 vektörlerde pay = **ortak tür sayısı**, paydadaki her kök = **o filmin tür sayısının karekökü**. Yani Toy Story (5 tür) ile Jumanji (3 tür), ortak 3: $3 / (\sqrt5 \cdot \sqrt3) = 0{,}775$.

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">
<b>Yanıt</b>
Toy Story ile Jumanji'nin **3 ortak türü** var (Adventure, Children, Fantasy). Toy Story ile GoldenEye'ın **1** (Adventure). Toy Story 5, Jumanji 3, GoldenEye 3 türlü. Bunlar aşağıdaki formülün payı: $3/\sqrt{15}$ ve $1/\sqrt{15}$.
</div>
</details>

**✍️ Kendin yaz:** `kosinus(a, b)` fonksiyonu. `a` ve `b` birer satır (`mini.loc["Toy Story"]` gibi). İpucu: `(a * b).sum()`, `np.sqrt`.

In [ ]:
def kosinus(a, b):
    ...

print("Toy Story – Jumanji  :", round(kosinus(mini.loc["Toy Story"], mini.loc["Jumanji"]), 3))
print("Toy Story – GoldenEye:", round(kosinus(mini.loc["Toy Story"], mini.loc["GoldenEye"]), 3))
print("Heat – GoldenEye     :", round(kosinus(mini.loc["Heat"], mini.loc["GoldenEye"]), 3))

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Çözüm</b>

```python
def kosinus(a, b):
    return (a * b).sum() / (np.sqrt((a ** 2).sum()) * np.sqrt((b ** 2).sum()))
```
Toy Story – Jumanji 0,775 · Toy Story – GoldenEye 0,258 · Heat – GoldenEye 0,667. Elle hesapla karşılaştır: $3/\sqrt{15}$, $1/\sqrt{15}$, $2/\sqrt{9}$.

</div>

</details>

**🔮 Düşün:** Benzerliği **Öklid uzaklığı** ile de ölçebilirdik: $\sqrt{\sum_i (a_i - b_i)^2}$. 0/1 vektörlerde bu, **yalnız birinde olan** türlerin sayısının karekökü. Toy Story – Jumanji için kaç? Toy Story – GoldenEye için?

*Cevabım:* ...

In [ ]:
def oklid(a, b):
    return np.sqrt(((a - b) ** 2).sum())

for x, y in [("Toy Story", "Jumanji"), ("Toy Story", "GoldenEye"), ("Heat", "GoldenEye")]:
    print(f"{x} – {y}: kosinüs {kosinus(mini.loc[x], mini.loc[y]):.3f}  Öklid {oklid(mini.loc[x], mini.loc[y]):.3f}")

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#ffebee; color:#1a1a1a; border-left:6px solid #c62828; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Neden kosinüs?</b>

Öklid'de Toy Story – Jumanji $\sqrt2 = 1{,}41$ (Animation ve Comedy yalnız Toy Story'de), Toy Story – GoldenEye $\sqrt6 = 2{,}45$. Sıralama aynı çıktı; ama Öklid, **çok türü olan** filmleri her şeyden uzak gösterir: 8 türlü bir film, 2 türlü bir filme ortak 2 türde bile $\sqrt6$ uzaktadır. Kosinüs vektörün **uzunluğuna** değil **yönüne** bakar; bu yüzden metin ve tür gibi "kaç tane" sorusunun önemsiz olduğu verilerde kosinüs kullanılır. (Sunumdaki şekil: aynı doğrultuda iki ok, uzunlukları farklı; Öklid uzak der, kosinüs aynı der.)

</div>

</details>

---
## Bölüm 2: Bütün filmler (15 dk)
Beş filmde elle yaptığımızı 9.737 filmde kütüphaneyle yapalım. `cosine_similarity` her film çiftinin kosinüsünü tek seferde hesaplar.

**🔮 Tahmin et:** 9.737 film için benzerlik matrisi kaç satır × kaç sütun olur? Kaç sayı? (Her sayı 8 bayt; kaç MB?)

*Tahminim:* ...

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

n_film = len(filmler)
print(f"Tam matris: {n_film} × {n_film} = {n_film**2:,} sayı × 8 bayt = {n_film**2 * 8 / 1e6:,.0f} MB")

# O kadar belleği harcamayalım: ilk 5 film için deneyelim
ornek = cosine_similarity(turler.values[:5])
pd.DataFrame(ornek, index=filmler["title"][:5], columns=filmler["title"][:5]).round(2)

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e8f5e9; color:#1a1a1a; border-left:6px solid #2e7d32; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Kod ne yapıyor?</b>

`cosine_similarity(X)` X'in her satır çifti için kosinüsü hesaplar; Bölüm 1'deki fonksiyonun hızlı sürümü. Köşegen 1,0 (film kendine benzer). Tam matris 750 MB'ı aşar; bu yüzden aşağıda **tek film için** anlık hesaplayacağız: `cosine_similarity(bir_satir, bütün_tablo)` → 1 × 9.737.

</div>

</details>

**✍️ Kendin yaz:** `benzer_filmler(ad, n=10)`: verilen filmin tür satırını bütün tabloyla karşılaştır, kosinüsü en yüksek `n` filmi döndür (filmin kendisi hariç). İskelet hazır; iki boşluğu doldur.

In [ ]:
def benzer_filmler(ad, n=10):
    i = filmler.index[filmler["title"] == ad][0]                       # filmin satır numarası
    skor = cosine_similarity(..., ...)[0]                              # 1 × 9.737
    skor = pd.Series(skor, index=filmler["title"])
    return skor.drop(...).nlargest(n)

benzer_filmler("Toy Story (1995)")

**🔮 Düşün:** Listedeki filmlerin hepsinin kosinüsü 1,0. Neden? Bu liste "Toy Story'yi beğenene öner" diye gösterilse ne olur?

*Cevabım:* ...

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#ffebee; color:#1a1a1a; border-left:6px solid #c62828; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Aşırı özelleşme (over-specialization)</b>

Kosinüs 1,0 = **tamamen aynı tür kümesi**. Toy Story'nin beş türünü birebir taşıyan 12 film var; yöntem aralarında ayrım yapamaz, sıra rastgele. Üstelik hepsi çocuk animasyonu: kullanıcı başka hiçbir şey görmez. İçerik tabanlı yöntemin bilinen zayıflığı budur: **beğendiğinin aynısını önerir, şaşırtmaz.** Sepet analizi bunu yapmıyordu; Kill Bill 2'den Kill Bill 1'e gitti ama tür bilgisi kullanmadan, insanların davranışından.

İki iyileştirme (W10'da): beraberlik bozucu olarak **popülerlik** (kaç kişi puanlamış) ve türlere ek **etiketler** (tags.csv) ile daha zengin öğe profili.

</div>

</details>

**🔧 Değiştir:** Beraberlikleri bozmak için filmin aldığı puan sayısını kullan: kosinüsü eşit olanlar arasında çok puanlanan öne geçsin. Aşağıdaki hücre bunu yapıyor; `"Heat (1995)"` için de çalıştır.

In [ ]:
puan_sayisi = puanlar.merge(filmler[["movieId", "title"]]).groupby("title").size()   # film başına puan sayısı

def benzer_filmler2(ad, n=10):
    aday = benzer_filmler(ad, n=len(filmler) - 1)      # bütün filmler, kosinüse göre sıralı
    tablo = pd.DataFrame({"kosinus": aday, "puan_sayisi": puan_sayisi.reindex(aday.index).fillna(0)})
    return tablo.sort_values(["kosinus", "puan_sayisi"], ascending=False).head(n)

benzer_filmler2("Toy Story (1995)")

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#ffebee; color:#1a1a1a; border-left:6px solid #c62828; padding:10px 16px; border-radius:6px; margin:8px 0">
<b>Yorum</b>
`Heat (1995)` için de ilk 10 filmin kosinüsü 1,0 (Heat: Action, Crime, Thriller; aynı üç türü taşıyan çok film var). Puan sayısına göre sıralayınca Batman (1989, 189 puan), Die Hard (145), Die Hard: With a Vengeance (144), Kill Bill: Vol. 1 (131), Net, The (112) öne çıkar. Popülerlik yalnız **beraberliği bozar**; filmi türce daha benzer yapmaz. Hâlâ aynı tür kümesinden öneri geliyor.
</div>
</details>

---
## Bölüm 3: Kullanıcıdan profile (15 dk)
Şimdiye kadar "bu filme benzeyen" sorusunu sorduk. Asıl soru "**bu kullanıcıya** ne önerelim?" Yöntem: kullanıcının beğendiği filmlerin tür vektörlerinin **ortalaması** = **kullanıcı profili**. Sonra profil ile izlemediği filmler arasında kosinüs.

In [ ]:
KULLANICI = 1
begendi = puanlar[(puanlar["userId"] == KULLANICI) & (puanlar["rating"] >= 4)]
begendi_idx = filmler.index[filmler["movieId"].isin(begendi["movieId"])]

profil = turler.loc[begendi_idx].mean()          # her tür için: beğendiklerinin yüzde kaçı bu türde?
print(f"Kullanıcı {KULLANICI}: {len(begendi)} film beğenmiş")
profil.sort_values(ascending=False).head(8).round(2)

**🔮 Düşün:** Profil 20 sayıdan oluşan bir vektör; Toy Story'nin vektörüyle aynı boyda. Profilde "Action 0,38" ne demek? Profil bir "film" gibi davranabilir mi?

*Cevabım:* ...

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">
<b>Yanıt</b>
Profil, kullanıcının beğendiği filmlerin tür vektörlerinin **ortalaması**; film vektörleriyle aynı uzayda. "Action 0,38": beğendiği filmlerin %38'i Action. Profil gerçek bir film değil, "ortalama bir film"; ama aynı boyda olduğu için kosinüsle bütün filmlerle karşılaştırılabilir. İşte bu yüzden profil bir film gibi davranabilir.
</div>
</details>

**✍️ Kendin yaz:** `kullaniciya_oner(profil, kullanici, n=10)`: profil ile **bütün** filmler arasında kosinüs hesapla (`cosine_similarity(profil.values.reshape(1, -1), turler.values)`), kullanıcının **zaten puanladığı** filmleri çıkar (`kullanici`: kimin puanlarına bakılacağı), en yüksek `n` filmi döndür.

In [ ]:
def kullaniciya_oner(profil, kullanici, n=10):
    ...

kullaniciya_oner(profil, KULLANICI)

<details><summary><b>🔓 Çözdükten (ya da tartıştıktan) sonra aç</b></summary>

<div style="background:#e3f2fd; color:#1a1a1a; border-left:6px solid #1565c0; padding:10px 16px; border-radius:6px; margin:8px 0">

<b>Çözüm ve yorum</b>

```python
def kullaniciya_oner(profil, kullanici, n=10):
    skor = cosine_similarity(profil.values.reshape(1, -1), turler.values)[0]
    skor = pd.Series(skor, index=filmler["title"])
    izledi = filmler.loc[filmler["movieId"].isin(puanlar.loc[puanlar["userId"] == kullanici, "movieId"]), "title"]
    return skor.drop(izledi).nlargest(n)
```
Önerilenler, kullanıcının tür dağılımına en çok benzeyen tür karışımına sahip filmler. Yine beraberlikler var; yine popülerlikle bozulabilir. Dikkat: **düşük** puan verdiği filmler profile girmedi (eşik 4). Eşiği 3'e indirirsen profil nasıl değişir? Bu bir **karar**; projende gerekçelendirmen gereken türden.

</div>

</details>

---
## Kapanış
**Bugün:** Öğe profili (tür vektörü) → kosinüs benzerliği (elle ve kütüphaneyle) → "bu filme benzer" → kullanıcı profili → "bu kullanıcıya". Yöntem **ürünün ne olduğuna** bakıyor; sepet analizi **kimin ne aldığına** bakıyordu.

| | Sepet analizi | İçerik tabanlı |
|---|---|---|
| Girdi | Kim neyi birlikte aldı | Ürünün öznitelikleri |
| Yeni ürün | Kimse almadıysa öneremez | Öznitelikleri varsa önerir (soğuk başlangıç) |
| Zayıflık | Popüler ürün her yerde | Beğendiğinin aynısı, şaşırtmaz |

**Gelecek hafta (W10):** İşbirlikçi filtreleme. Ürüne bakmadan, "senin gibi beğenen insanlar şunu da beğendi" diyeceğiz; 4 kullanıcı × 4 film tablosunda benzerlik ağırlıklarını elle hesaplayacağız.

**Projen için soru:** Senin verinde öğelerin öznitelikleri var mı (kategori, açıklama, marka)? Yoksa içerik tabanlı yöntem senin verine uygulanamaz; bunu raporunda bir cümleyle yazabilmelisin.